# 02 — The group and its data (96 months)

**Goal (PROBLEM.md):** lay out *who is in the group* and build the *operational panel* every
decision reads — here over the full 8-year raw span (2018-04 → 2026-03).

**Pre-registered freeze:** the network is the 35 branches in `config/nhs_network.json`,
selected and audited on the **frozen window 2021-04..2026-03** (notebook 01). Extra history
deepens each branch's series; it never re-selects branches. Completeness is asserted on the
frozen window only — pre-2021 months MAY be partial for some branches.

In [ ]:
import json
import glob
import os
import pandas as pd

# the org chart: regions -> branches (with board + departments + typical demand)
net = json.load(open("../config/nhs_network.json"))
rows = [{"region": r["region"], "branch": t["code"], "name": t["name"].title()[:44],
         "board_icb": t["icb"], "departments": t["specialties"],
         "demand_per_month": t["mean_monthly_referrals"]}
        for r in net["regions"] for t in r["trusts"]]
org = pd.DataFrame(rows)
print(f"group: {org['branch'].nunique()} branches | {org['region'].nunique()} regions | "
      f"{org['board_icb'].nunique()} boards | ~{org['demand_per_month'].sum():,} referrals/month")

In [ ]:
# build the operational panel from ALL raw files on disk (96 on the A100; ~10 min)
OUT_CSV = "../data/processed/rtt_long.csv"
ID_COLS = ["Period", "Provider Org Code", "Provider Org Name", "RTT Part Type",
           "Treatment Function Code", "Treatment Function Name"]
BAND_COLS = [f"Gt {i:02d} To {i + 1:02d} Weeks SUM 1" for i in range(104)] + ["Gt 104 Weeks SUM 1"]
TOTAL_COLS = ["Total", "Total All"]  # counts live in Total All (Total empty for Part_2/2A/3)
codes = set(org["branch"])
monthly = []
for path in sorted(glob.glob("../data/nhs_rtt_referral_to_treatment/rtt_*.csv")):
    month = os.path.basename(path)[4:11]
    df = pd.read_csv(path, usecols=ID_COLS + BAND_COLS + TOTAL_COLS, dtype={c: str for c in ID_COLS})
    df = df[df["Treatment Function Code"] != "C_999"]           # drop double-counting totals
    df = df[df["Provider Org Code"].isin(codes)]                 # network branches only
    df[BAND_COLS + TOTAL_COLS] = df[BAND_COLS + TOTAL_COLS].apply(pd.to_numeric, errors="coerce").fillna(0)
    df = df.assign(weeks_0_18=df[BAND_COLS[:18]].sum(axis=1),
                   weeks_18_52=df[BAND_COLS[18:52]].sum(axis=1),
                   weeks_52_plus=df[BAND_COLS[52:]].sum(axis=1))
    out = df.groupby(["Provider Org Code", "Provider Org Name", "RTT Part Type",
                      "Treatment Function Code", "Treatment Function Name"], as_index=False)[
        ["weeks_0_18", "weeks_18_52", "weeks_52_plus", "Total", "Total All"]].sum()
    out.insert(0, "month", month)
    out.columns = ["month", "provider_code", "provider_name", "part", "specialty_code",
                   "specialty_name", "weeks_0_18", "weeks_18_52", "weeks_52_plus", "total", "total_all"]
    monthly.append(out)
    print(month, end=" ", flush=True)
long = pd.concat(monthly, ignore_index=True)
os.makedirs(os.path.dirname(OUT_CSV), exist_ok=True)
long.to_csv(OUT_CSV, index=False)
print(f"\nbuilt {OUT_CSV}: {len(long)} rows")

In [ ]:
# checks: completeness on the FROZEN window; history coverage reported, not asserted
WINDOW = ("2021-04", "2026-03")
assert long["provider_code"].nunique() == 35 and set(long["provider_code"]) == set(org["branch"])
assert set(long["part"]) == {"Part_1A", "Part_1B", "Part_2", "Part_2A", "Part_3"}
core = long[long["month"].between(*WINDOW)]
presence = core.groupby("provider_code")["month"].nunique()
assert (presence == core["month"].nunique()).all(), "a branch is missing frozen-window months!"
print(f"panel OK: {long['month'].nunique()} months ({long['month'].min()}..{long['month'].max()}) | "
      f"frozen window complete for all 35 branches | {long['specialty_code'].nunique()} departments")
print("\nearliest month per branch (history coverage):")
print(long.groupby("provider_code")["month"].min().value_counts().sort_index().to_string())